# molten-arc
### codename: BACON — Battery Architectures via Computational Observation of Nonaqueous salts

This notebook is the molten salt counterpart to the Na-ion electrolyte work in `SolvationNet/notebooks/na-ions/`. Where the Na work focuses on room-temperature organic electrolytes (DME, PC + NaPF₆), here we're dealing with **pure ionic melts** at high temperatures — the kind used in aluminium-ion batteries, liquid metal batteries, and molten-air batteries.

The starting systems are drawn from the review by Liu et al., *Materials Today* 60 (2022) 128–157.

### What's different from the Na electrolyte boxes?

1. **No organic solvents.** The entire simulation cell is ions — cations and anions, that's it.
2. **High temperatures.** We're simulating at 120–700 °C instead of 25 °C. The thermostat matters more.
3. **Density from experiment.** Instead of computing density from a known solvent + salt molarity, we set the density directly from literature values for the molten salt mixture.
4. **Ion PDB files are simpler.** Single atoms (Na⁺, Li⁺, Cl⁻, etc.) or small polyatomic ions (AlCl₄⁻, NO₃⁻).

### Workflow

Same pipeline as the Na work, just adapted for molten salts:

```
Define composition  →  Build ion PDBs  →  Packmol  →  Equilibrate (NVT → NPT)  →  Production MD  →  Analyze
```

For batch runs on Perlmutter, see `bacon.py` at the project root.

---
## 1. The Systems

We have two categories of molten salts in the database so far:

### Electrolyte Salts (used *as* the electrolyte)

| System | Application | Working Temp | Key Feature |
|--------|------------|-------------|-------------|
| AlCl₃–NaCl (62:48) | Al-ion battery | 120 °C | AlCl₄⁻/Al₂Cl₇⁻ equilibrium |
| AlCl₃–NaCl–LiCl–KCl | Al-ion battery | 90 °C | Lowest-temp AIB electrolyte |
| LiCl–LiF (70:30) | Liquid metal battery | 550 °C | Li metal negative electrode |
| LiCl–LiI (36:64) | Liquid metal battery | 410 °C | Li‖Bi cell, Li₃Bi alloy |
| LiCl–NaCl–CaCl₂ | Liquid metal battery | 600 °C | Ca–Bi positive electrode |
| MgCl₂–NaCl–KCl | Liquid metal battery | 700 °C | Mg‖Sb, earth-abundant |
| LiF–LiCl–LiBr | Liquid metal battery | 500 °C | Li‖Sb–Pb system |
| Li₂CO₃–Na₂CO₃–K₂CO₃–LiOH | Molten-air battery | 500 °C | Fe electrode, O²⁻ conductor |

### Synthesis Salts (reaction media for making electrode materials)

| Salt | Typical Temp | What it makes |
|------|-------------|---------------|
| KCl | 800–1000 °C | LiCoO₂ polyhedra, NCM single crystals, Li-rich oxides |
| NaCl–KCl | 700–900 °C | LiMn₂O₄ spinel, LiFePO₄ crystals |
| LiNO₃–LiOH | 250–500 °C | Nano-sheets, nano-petals, low-temp synthesis |
| LiCl–KCl | 400–900 °C | Li₄Ti₅O₁₂, spinel LNMO, Si nanoparticles |
| AlCl₃ | 200–250 °C | Si from metallothermic reduction |

---
## 2. Building the Ion Coordinates

For the Na-electrolyte work we needed PDB files of organic molecules (DME, PC) plus ions (Na⁺, PF₆⁻). For molten salts, our building blocks are simpler — mostly monatomic ions. But some systems have polyatomic species like AlCl₄⁻, NO₃⁻, or CO₃²⁻.

We'll generate PDB files for the ions we need.

In [ ]:
import os
import math
import numpy as np

ION_DIR = "../data/ions"
os.makedirs(ION_DIR, exist_ok=True)

# Monatomic ions — just a single HETATM line each
MONATOMIC_IONS = {
    "Li":  ("LI",  "Li", +1),
    "Na":  ("NA",  "Na", +1),
    "K":   ("K",   "K",  +1),
    "Ca":  ("CA",  "Ca", +2),
    "Mg":  ("MG",  "Mg", +2),
    "Ba":  ("BA",  "Ba", +2),
    "Al":  ("AL",  "Al", +3),
    "Zn":  ("ZN",  "Zn", +2),
    "Cl":  ("CL",  "Cl", -1),
    "F":   ("F",   "F",  -1),
    "Br":  ("BR",  "Br", -1),
    "I":   ("I",   "I",  -1),
}

for name, (resname, element, charge) in MONATOMIC_IONS.items():
    pdb_path = os.path.join(ION_DIR, f"{name.lower()}.pdb")
    with open(pdb_path, "w") as f:
        f.write(f"HETATM    1 {element:>2s}   {resname:>3s} A   1"
                f"       0.000   0.000   0.000  1.00  0.00"
                f"          {element:>2s}\n")
        f.write("END\n")

print(f"Generated {len(MONATOMIC_IONS)} monatomic ion PDB files in {ION_DIR}/")
print("Files:", ", ".join(f"{n.lower()}.pdb" for n in MONATOMIC_IONS))

For polyatomic ions like NO₃⁻, OH⁻, and CO₃²⁻ we need proper 3D coordinates. We can either grab them from PubChem or build them with RDKit.

In [ ]:
# Polyatomic ions — built from SMILES
# These show up in synthesis salts (LiNO3, LiOH, carbonates)
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem

    POLYATOMIC_IONS = {
        "NO3": {"smiles": "[O-][N+](=O)[O-]",  "resname": "NO3"},
        "OH":  {"smiles": "[OH-]",              "resname": "OH"},
        "CO3": {"smiles": "[O-]C(=O)[O-]",     "resname": "CO3"},
    }

    for name, info in POLYATOMIC_IONS.items():
        mol = Chem.MolFromSmiles(info["smiles"])
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        pdb_path = os.path.join(ION_DIR, f"{name.lower()}.pdb")
        Chem.MolToPDBFile(mol, pdb_path)
        print(f"  {name:>4s} -> {pdb_path}")

    print(f"Generated {len(POLYATOMIC_IONS)} polyatomic ion PDB files.")

except ImportError:
    print("RDKit not available — skip polyatomic ions or install it.")
    print("  pip install rdkit")

---
## 3. Calculating Ion Counts

This is the key calculation that's different from the Na work. In the Na electrolyte boxes, we started with a known solvent density and a target salt molarity. Here, the **whole box is the salt**, so we go from:

$$N_{\text{formula units}} = \frac{\rho \cdot V_{\text{box}} \cdot N_A}{\bar{M}}$$

where $\bar{M}$ is the mole-fraction-weighted average molar mass of the salt mixture.

Then we split those formula units according to the mol% of each component and dissociate into individual ions.

### Example: NaAlCl₄ for Al-ion batteries

AlCl₃ : NaCl = 62 : 48 mol%  
Working temperature: 120 °C  
Estimated density: ~1.6 g/cm³

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from molten_salt_systems import ELECTROLYTE_SYSTEMS, SALT_MOLAR_MASSES
from pack_molten_salt import compute_ion_counts

# Let's work through the NaAlCl4 system
system = ELECTROLYTE_SYSTEMS["NaAlCl4"]
print(f"System: {system['description']}")
print(f"Application: {system['application']}")
print(f"Working temp: {system['working_temp_C']} °C")
print(f"Components: {system['components']}")
print()

box_size = 25.0  # Å
density = 1.6    # g/cm³, approximate for NaAlCl4 near 120 °C

salt_counts, ion_counts = compute_ion_counts(
    system["components"], box_size, density
)

print(f"Box: {box_size} Å  ({box_size**3:.0f} ų)")
print(f"Target density: {density} g/cm³")
print(f"\nFormula units:")
for salt, count in salt_counts.items():
    print(f"  {salt:>8s}: {count:>4d}")

print(f"\nIon counts:")
total = 0
for ion, count in ion_counts.items():
    print(f"  {ion:>8s}: {count:>4d}")
    total += count
print(f"  {'Total':>8s}: {total:>4d} ions")

### Charge neutrality check

Molten salts must be charge-neutral. Let's verify.

In [ ]:
ION_CHARGES = {
    "Li": +1, "Na": +1, "K": +1, "Ca": +2, "Mg": +2,
    "Ba": +2, "Al": +3, "Zn": +2,
    "Cl": -1, "F": -1, "Br": -1, "I": -1,
    "NO3": -1, "OH": -1, "CO3": -2,
}

total_charge = sum(ION_CHARGES.get(ion, 0) * count
                   for ion, count in ion_counts.items())

print(f"Total charge: {total_charge:+d}")
if total_charge == 0:
    print("Charge neutral — good to go.")
else:
    print(f"WARNING: System is not neutral! Off by {total_charge}.")
    print("  You'll need to adjust ion counts to fix this.")

---
## 4. Packing with Packmol

Same idea as the Na work — we write a Packmol input and run it. The tolerance is bumped up slightly (2.5 Å instead of 2.0 Å) because ionic radii in molten salts are larger than covalent bond lengths.

### Using the script directly

```bash
python ../scripts/pack_molten_salt.py \
  --system NaAlCl4 \
  --box-size 25 \
  --density 1.6 \
  --ion-dir ../data/ions \
  --output ../data/packed/NaAlCl4_25A.pdb
```

Or do it right here in the notebook:

In [ ]:
import subprocess, shutil

PACKED_DIR = "../data/packed"
os.makedirs(PACKED_DIR, exist_ok=True)

output_pdb = os.path.join(PACKED_DIR, "NaAlCl4_25A.pdb")

# Build the packmol input
packmol_lines = [
    "tolerance 2.5",
    "filetype pdb",
    f"output {os.path.abspath(output_pdb)}",
    "seed 42",
    "",
]

for ion_name, count in ion_counts.items():
    pdb_file = os.path.abspath(os.path.join(ION_DIR, f"{ion_name.lower()}.pdb"))
    packmol_lines.append(f"structure {pdb_file}")
    packmol_lines.append(f"  number {count}")
    packmol_lines.append(f"  inside box 0. 0. 0. {box_size} {box_size} {box_size}")
    packmol_lines.append("end structure")
    packmol_lines.append("")

packmol_input = "\n".join(packmol_lines)
print("--- Packmol input ---")
print(packmol_input)

if shutil.which("packmol"):
    result = subprocess.run(
        ["packmol"], input=packmol_input,
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # Add CRYST1 header for periodic boundary conditions
        from pack_molten_salt import add_cryst1_to_pdb
        add_cryst1_to_pdb(output_pdb, box_size)
        print(f"\nPacked cell written to: {output_pdb}")
    else:
        print(f"Packmol failed: {result.stderr[-300:]}")
else:
    print("\nPackmol not installed. Install with:")
    print("  conda install -c conda-forge packmol")
    print("  or: pip install packmol")

---
## 5. Equilibration and MD

This follows the same Langevin / NPT protocol as the Na-electrolyte work, but with key differences for molten salts:

### Temperature
We're running at the working temperature of the salt system. For NaAlCl₄ that's 120 °C (393 K). For LMB electrolytes it could be 500–700 °C (773–973 K). The thermostat coupling needs to be tighter at high temperatures to prevent runaway.

### Barostat
NPT at 1 atm, same as before. The density will adjust to whatever the force field predicts — a good sanity check against the experimental value we used for packing.

### Force field
The UMA/OrbMol universal potentials should handle inorganic ionic systems. For production-quality molten salt simulations you might want specialized potentials (Born-Mayer-Huggins, Polarizable Ion Model), but for exploratory work the universal MLIPs are a good starting point.

### Timestep
1.0 fs is fine for heavy ions at high temperature. The Na work used 0.5–1.0 fs.

In [ ]:
# This cell sets up the equilibration — same pattern as the Na electrolyte work
# but at 393 K (120 °C) for NaAlCl4
#
# For actual runs, use bacon.py for Perlmutter batch submission.
# This is here to show the setup.

EQUILIBRATION_TEMPLATE = """
# BACON — molten salt equilibration: {system_name}
# Temperature: {temp_K:.1f} K ({temp_C} °C)
# Box: {box_size} Å, {n_ions} ions

from ase.io import read
from ase.md.langevin import Langevin
from ase.md.npt import NPT
from ase.io import Trajectory
from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
import torch

# 1. Load the packed cell
atoms = read("{input_pdb}")
atoms.set_cell([{box_size}, {box_size}, {box_size}])
atoms.set_pbc(True)

# 2. Attach the calculator (UMA or OrbMol)
from fairchem.core import pretrained_mlip, FAIRChemCalculator
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device="cuda")
atoms.calc = FAIRChemCalculator(predictor, task_name="omol")

# 3. Initialize velocities at target temperature
MaxwellBoltzmannDistribution(atoms, temperature_K={temp_K})

# 4. NVT equilibration (let the ions settle before turning on the barostat)
nvt = Langevin(
    atoms,
    timestep=1.0 * units.fs,
    temperature_K={temp_K},
    friction=0.01 / units.fs,
)
nvt_traj = Trajectory("nvt_{system_name}.traj", "w", atoms)
nvt.attach(nvt_traj.write, interval=10)
nvt.run(steps=5000)  # 5 ps NVT warmup

# 5. NPT production (Parrinello-Rahman)
npt = NPT(
    atoms=atoms,
    timestep=1.0 * units.fs,
    temperature_K={temp_K},
    externalstress=1.0 * units.bar,
    ttime=100 * units.fs,
    pfactor=0.1,
    mask=([[1,0,0],[0,1,0],[0,0,1]]),
)
npt_traj = Trajectory("npt_{system_name}.traj", "w", atoms)
npt.attach(npt_traj.write, interval=10)
npt.run(steps=50000)  # 50 ps NPT
"""

temp_C = system["working_temp_C"]
temp_K = temp_C + 273.15

script = EQUILIBRATION_TEMPLATE.format(
    system_name="NaAlCl4",
    temp_C=temp_C,
    temp_K=temp_K,
    box_size=box_size,
    n_ions=total,
    input_pdb=output_pdb,
)

print(f"Equilibration script for NaAlCl4 at {temp_K:.0f} K ({temp_C} °C):")
print(f"  {total} ions in a {box_size} Å box")
print(f"  NVT warmup: 5 ps")
print(f"  NPT production: 50 ps")
print()
print("To run this on Perlmutter:")
print("  python bacon.py campaign create my_salt_run \\")
print("    --system NaAlCl4 --box-size 25 --density 1.6")

---
## 6. All the Systems at a Glance

Let's compute ion counts for every electrolyte system in our database, so we know what we're working with.

In [ ]:
from molten_salt_systems import ELECTROLYTE_SYSTEMS, MOLTEN_SALT_DENSITIES
from pack_molten_salt import compute_ion_counts

# Rough density estimates for each system (g/cm³)
# These are approximate — real values depend on composition and temperature
DENSITY_ESTIMATES = {
    "NaAlCl4":                  1.60,
    "AlCl3-NaCl-LiCl-KCl":     1.55,
    "AlCl3-NaCl-KCl":          1.58,
    "LiCl-LiF":                1.65,
    "LiCl-LiI":                2.50,
    "LiCl-NaCl-CaCl2":         1.90,
    "LiCl-NaCl-CaCl2-BaCl2":   2.10,
    "MgCl2-NaCl-KCl":          1.70,
    "LiCl-NaCl-KCl":           1.60,
    "LiF-LiCl-LiBr":           2.20,
    "Li2CO3-Na2CO3-K2CO3-LiOH": 1.90,
}

BOX = 25.0  # Å

print(f"{'System':<32s} {'App':>4s} {'T(°C)':>6s} {'ρ':>5s} {'Ions':>5s}")
print("=" * 60)

for name, sys in ELECTROLYTE_SYSTEMS.items():
    rho = DENSITY_ESTIMATES.get(name, 1.8)
    _, ions = compute_ion_counts(sys["components"], BOX, rho)
    n_total = sum(ions.values())
    print(f"{name:<32s} {sys['application']:>4s} {sys['working_temp_C']:>5d}  {rho:>5.2f} {n_total:>5d}")

---
## 7. What to Look for in the Simulations

Once we have equilibrated boxes, the key observables for molten salt electrolytes are:

### Transport properties
- **Ionic conductivity** — from the Green-Kubo relation or Nernst-Einstein equation. Compare to the experimental values in Table 3.
- **Self-diffusion coefficients** — MSD of each ion species. In molten salts, cations and anions can have very different mobilities.
- **Viscosity** — via the stress autocorrelation function. Low viscosity = better electrolyte.

### Structure
- **Radial distribution functions** — cation-anion, cation-cation, anion-anion pairs. Look for coordination numbers and nearest-neighbor distances.
- **Coordination environment** — how many Cl⁻ surround each Al³⁺? This tells you whether you have free Al³⁺, AlCl₄⁻, or Al₂Cl₇⁻.

### Thermodynamics
- **Density** — does the NPT simulation converge to the experimental value?
- **Heat capacity** — from energy fluctuations.

### For battery design specifically
- **Metal solubility** — do electrode metals dissolve into the melt? (Self-discharge in LMBs)
- **Electrochemical window** — which species oxidize/reduce first?
- **Speciation** — in AlCl₃-NaCl, the AlCl₄⁻ / Al₂Cl₇⁻ ratio controls the battery mechanism.

---
## References

1. H. Liu, X. Zhang, S. He, D. He, Y. Shang, H. Yu. **Molten salts for rechargeable batteries.** *Materials Today* 60, 128–157 (2022). [doi:10.1016/j.mattod.2022.09.005](https://doi.org/10.1016/j.mattod.2022.09.005)

2. G.J. Janz, R.P.T. Tomkins, C.B. Allen. **Molten salts: Volume 4, Part 2, Chlorides and mixtures.** *J. Phys. Chem. Ref. Data* 4 (1975). — Standard source for molten salt densities and transport properties.

3. H. Kim et al. **Liquid metal batteries: Past, present, and future.** *Chem. Rev.* 113, 2075 (2013). — The definitive LMB review.

4. Y. Song et al. **A low-cost and high-performance aluminum-ion battery based on a molten salt electrolyte.** *J. Mater. Chem. A* 5, 1282 (2017). — NaAlCl₄ electrolyte for AIBs.